我很喜欢课程里的这句话：
> 市面上已有LangChain、LlamaIndex等众多优秀框架，为何还要“重复造轮子”？
> 答案在于，尽管成熟的框架在工程效率上优势显著，但直接使用高度抽象的工具，并不利于我们了解背后的设计机制是怎么运行的，或者是有何好处。
> 其次，这个过程会暴露出项目的工程挑战。框架为我们处理了许多问题，例如模型输出格式的解析、工具调用失败的重试、防止智能体陷入死循环等。亲手处理这些问题，是培养系统设计能力的最直接方式。
> 最后，也是最重要的一点，掌握了设计原理，你才能真正地从一个框架的“使用者”转变为一个智能体应用的“创造者”。当标准组件无法满足你的复杂需求时，你将拥有深度定制乃至从零构建一个全新智能体的能力。

| 范式             | 核心思想                 | 适合场景               |
| -------------- | -------------------- | ------------------ |
| ReAct          | 边思考边行动，行动后观察结果，再继续思考 | 搜索、查资料、调用 API、动态任务 |
| Plan-and-Solve | 先完整规划，再按步骤执行         | 结构清晰、多步骤推理任务       |
| Reflection     | 先生成初稿，再自我批判和优化       | 代码生成、写作、复杂方案优化     |

## 2.1 封装基础 LLM 客户端类
为了让代码结构更清晰、更易于复用，我们来定义一个专属的LLM客户端类。这个类将封装所有与模型服务交互的细节，让我们的主逻辑可以更专注于智能体的构建。

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from typing import List, Dict

# 加载 .env 文件中的环境变量
load_dotenv()

class HelloAgentsLLM:
    """
    为本书 "Hello Agents" 定制的LLM客户端。
    它用于调用任何兼容OpenAI接口的服务，并默认使用流式响应。
    """
    def __init__(self, model: str = None, apiKey: str = None, baseUrl: str = None, timeout: int = None):
        """
        初始化客户端。优先使用传入参数，如果未提供，则从环境变量加载。
        """
        self.model = model or os.getenv("LLM_MODEL_ID")
        apiKey = apiKey or os.getenv("LLM_API_KEY")
        baseUrl = baseUrl or os.getenv("LLM_BASE_URL")
        timeout = timeout or int(os.getenv("LLM_TIMEOUT", 60))
        
        if not all([self.model, apiKey, baseUrl]):
            raise ValueError("模型ID、API密钥和服务地址必须被提供或在.env文件中定义。")

        self.client = OpenAI(api_key=apiKey, base_url=baseUrl, timeout=timeout)

    def think(self, messages: List[Dict[str, str]], temperature: float = 0) -> str:
        """
        调用大语言模型进行思考，并返回其响应。
        """
        print(f"正在调用 {self.model} 模型...")
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
                stream=True,
            )
            
            # 处理流式响应
            print("大语言模型响应成功:")
            collected_content = []
            for chunk in response:
                if not chunk.choices:
                    continue
                content = chunk.choices[0].delta.content or ""
                print(content, end="", flush=True)
                collected_content.append(content)
            print()  # 在流式输出结束后换行
            return "".join(collected_content)

        except Exception as e:
            print(f"调用LLM API时发生错误: {e}")
            return None

# --- 客户端使用示例 ---
if __name__ == '__main__':
    try:
        llmClient = HelloAgentsLLM()
        
        exampleMessages = [
            {"role": "system", "content": "You are a helpful assistant that writes Python code."},
            {"role": "user", "content": "写一个快速排序算法"}
        ]
        
        print("--- 调用LLM ---")
        responseText = llmClient.think(exampleMessages)
        if responseText:
            print("\n\n--- 完整模型响应 ---")
            print(responseText)

    except ValueError as e:
        print(e)

--- 调用LLM ---
正在调用 deepseek-v4-flash 模型...
大语言模型响应成功:
以下是快速排序算法的Python实现，采用原地排序（in-place）和递归方式，并以最后一个元素作为基准（pivot）。代码包含分区函数和主排序函数，并附有测试示例。

```python
def partition(arr, low, high):
    """
    分区函数：将数组分为小于基准和大于基准两部分，返回基准的最终位置。
    """
    pivot = arr[high]          # 选择最后一个元素作为基准
    i = low - 1                # i 指向最后一个小于等于基准的元素
    for j in range(low, high):
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]
    # 将基准放到正确的位置（i+1）
    arr[i + 1], arr[high] = arr[high], arr[i + 1]
    return i + 1

def quicksort(arr, low, high):
    """递归快速排序"""
    if low < high:
        pi = partition(arr, low, high)   # 获取基准位置
        quicksort(arr, low, pi - 1)      # 排序左半部分
        quicksort(arr, pi + 1, high)     # 排序右半部分

def quick_sort(arr):
    """对外包装函数，自动传入边界"""
    quicksort(arr, 0, len(arr) - 1)
    return arr

# 测试
if __name__ == "__main__":
    test_arr = [10, 7, 8, 9, 1, 5]
    print("原始数组:", test_arr)
    sorted_arr = quick_sort(test_arr)
    pri

## 2.2 ReAct
之前24年网页版就有这样的区分：
- 快速模式：直接输出要执行的动作，但缺乏规划和纠错能力
- 思考模式：引导模型进行复杂的逻辑推理，但无法与外部世界交互，容易产生事实幻觉

ReAct 全称是 Reasoning and Acting，核心是把 推理 Reasoning 和 行动 Acting 结合起来，用固定轨迹组织模型重复 **Thought → Action → Observation** 的 Loop。
```
Thought: 我需要先知道当前信息。
Action: 调用搜索工具。
Observation: 工具返回搜索结果。
Thought: 根据搜索结果，我还需要进一步确认。
Action: 再调用工具。
Observation: 得到补充信息。
Final Answer: 输出最终答案。
```

这种机制特别适用于以下场景：

- 需要外部知识的任务：如查询实时信息（天气、新闻、股价）、搜索专业领域的知识等。
- 需要精确计算的任务：将数学问题交给计算器工具，避免LLM的计算错误。
- 需要与API交互的任务：如操作数据库、调用某个服务的API来完成特定功能。
### 2.2.1 Tools
为了让ReAct范式能够真正解决我们设定的问题，智能体需要具备调用外部工具的能力。
一个良好定义的工具应包含以下三个核心要素：

1. **名称 (Name)**： 一个简洁、唯一的标识符，供智能体在 `Action` 中调用，例如 `Search`。
2. **描述 (Description)**： 一段清晰的自然语言描述，说明这个工具的用途。**这是整个机制中最关键的部分**，因为大语言模型会依赖这段描述来判断何时使用哪个工具。
3. **执行逻辑 (Execution Logic)**： 真正执行任务的函数或方法。

In [ ]:
from serpapi import SerpApiClient

def search(query: str) -> str:
    """
    一个基于SerpApi的实战网页搜索引擎工具。
    它会智能地解析搜索结果，优先返回直接答案或知识图谱信息。
    """
    print(f"🔍 正在执行 [SerpApi] 网页搜索: {query}")
    try:
        api_key = os.getenv("SERPAPI_API_KEY")
        if not api_key:
            return "错误:SERPAPI_API_KEY 未在 .env 文件中配置。"

        params = {
            "engine": "google",
            "q": query,
            "api_key": api_key,
            "gl": "cn",  # 国家代码
            "hl": "zh-cn", # 语言代码
        }
        
        client = SerpApiClient(params)
        results = client.get_dict()
        
        # 智能解析:优先寻找最直接的答案
        if "answer_box_list" in results:
            return "\n".join(results["answer_box_list"])
        if "answer_box" in results and "answer" in results["answer_box"]:
            return results["answer_box"]["answer"]
        if "knowledge_graph" in results and "description" in results["knowledge_graph"]:
            return results["knowledge_graph"]["description"]
        if "organic_results" in results and results["organic_results"]:
            # 如果没有直接答案，则返回前三个有机结果的摘要
            snippets = [
                f"[{i+1}] {res.get('title', '')}\n{res.get('snippet', '')}"
                for i, res in enumerate(results["organic_results"][:3])
            ]
            return "\n\n".join(snippets)
        
        return f"对不起，没有找到关于 '{query}' 的信息。"

    except Exception as e:
        return f"搜索时发生错误: {e}"

> 这里因为代理问题，等我们能成功注册 [SerpApi官网](https://serpapi.com/) 账户时再回来补充

在上述代码中，首先会检查是否存在 `answer_box`（Google的答案摘要框）或 `knowledge_graph`（知识图谱）等信息，如果存在，就直接返回这些最精确的答案。如果不存在，它才会退而求其次，返回前三个常规搜索结果的摘要。这种“智能解析”能为LLM提供质量更高的信息输入。

### 2.2.2 工具执行器
当智能体需要使用多种工具时（例如，除了搜索，还可能需要计算、查询数据库等），我们需要一个统一的管理器来注册和调度这些工具。为此，我们创建一个 `ToolExecutor` 类。


In [ ]:
from typing import Dict, Any

class ToolExecutor:
    """
    一个工具执行器，负责管理和执行工具。
    """
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def registerTool(self, name: str, description: str, func: callable):
        """
        向工具箱中注册一个新工具。
        """
        if name in self.tools:
            print(f"警告:工具 '{name}' 已存在，将被覆盖。")
        self.tools[name] = {"description": description, "func": func}
        print(f"工具 '{name}' 已注册。")

    def getTool(self, name: str) -> callable:
        """
        根据名称获取一个工具的执行函数。
        """
        return self.tools.get(name, {}).get("func")

    def getAvailableTools(self) -> str:
        """
        获取所有可用工具的格式化描述字符串。
        """
        return "\n".join([
            f"- {name}: {info['description']}" 
            for name, info in self.tools.items()
        ])

In [ ]:
if __name__ == '__main__':
    # 1. 初始化工具执行器
    toolExecutor = ToolExecutor()

    # 2. 注册我们的实战搜索工具
    search_description = "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
    toolExecutor.registerTool("Search", search_description, search)
    
    # 3. 打印可用的工具
    print("\n--- 可用的工具 ---")
    print(toolExecutor.getAvailableTools())

    # 4. 智能体的Action调用，这次我们问一个实时性的问题
    print("\n--- 执行 Action: Search['英伟达最新的GPU型号是什么'] ---")
    tool_name = "Search"
    tool_input = "英伟达最新的GPU型号是什么"

    tool_function = toolExecutor.getTool(tool_name)
    if tool_function:
        observation = tool_function(tool_input)
        print("--- 观察 (Observation) ---")
        print(observation)
    else:
        print(f"错误:未找到名为 '{tool_name}' 的工具。")

### 2.2.3 系统提示词设计
现在，我们将所有独立的组件，LLM客户端和工具执行器组装起来，构建一个完整的 ReAct 智能体。我们将通过一个 `ReActAgent` 类来封装其核心逻辑。

首先自然是提示词，整个 ReAct 机制的基石，它为大语言模型提供了行动的操作指令。我们需要精心设计一个模板，它将动态地插入可用工具、用户问题以及中间步骤的交互历史。

In [ ]:
REACT_PROMPT_TEMPLATE = """
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下:
{tools}

请严格按照以下格式进行回应:

Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，必须是以下格式之一:
- `{{tool_name}}[{{tool_input}}]`:调用一个可用工具。
- `Finish[最终答案]`:当你认为已经获得最终答案时。
- 当你收集到足够的信息，能够回答用户的最终问题时，你必须在Action:字段后使用 Finish[最终答案] 来输出最终答案。

现在，请开始解决以下问题:
Question: {question}
History: {history}
"""

这个模板定义了智能体与LLM之间交互的规范：

- **角色定义**： “你是一个有能力调用外部工具的智能助手”，设定了LLM的角色。
- **工具清单 (`{tools}`)**： 告知LLM它有哪些可用的“手脚”。
- **格式规约 (`Thought`/`Action`)**： 这是最重要的部分，它强制LLM的输出具有结构性，使我们能通过代码精确解析其意图。
- **动态上下文 (`{question}`/`{history}`)**： 将用户的原始问题和不断累积的交互历史注入，让LLM基于完整的上下文进行决策。

### 2.2.4 核心循环
“格式化提示词 -> 调用LLM -> 执行动作 -> 整合结果”，直到任务完成或达到最大步数限制。

In [ ]:
import re
class ReActAgent:
    def __init__(self, llm_client: HelloAgentsLLM, tool_executor: ToolExecutor, max_steps: int = 5):
        self.llm_client = llm_client
        self.tool_executor = tool_executor
        self.max_steps = max_steps
        self.history = []

    def run(self, question: str):
        """
        运行ReAct智能体来回答一个问题。
        """
        self.history = [] # 每次运行时重置历史记录
        current_step = 0

        while current_step < self.max_steps:
            current_step += 1
            print(f"--- 第 {current_step} 步 ---")

            # 1. 格式化提示词
            tools_desc = self.tool_executor.getAvailableTools()
            history_str = "\n".join(self.history)
            prompt = REACT_PROMPT_TEMPLATE.format(
                tools=tools_desc,
                question=question,
                history=history_str
            )

            # 2. 调用LLM进行思考
            messages = [{"role": "user", "content": prompt}]
            response_text = self.llm_client.think(messages=messages)
            
            if not response_text:
                print("错误:LLM未能返回有效响应。")
                break
            # 3. 解析LLM的输出
            thought, action = self._parse_output(response_text)
            
            if thought:
                print(f"思考: {thought}")

            if not action:
                print("警告:未能解析出有效的Action，流程终止。")
                break

            # 4. 执行Action
            if action.startswith("Finish"):
                # 如果是Finish指令，提取最终答案并结束
                final_answer = re.match(r"Finish\[(.*)\]", action).group(1)
                print(f"🎉 最终答案: {final_answer}")
                return final_answer
            
            tool_name, tool_input = self._parse_action(action)
            if not tool_name or not tool_input:
                # ... 处理无效Action格式 ...
                continue

            print(f"🎬 行动: {tool_name}[{tool_input}]")
            
            tool_function = self.tool_executor.getTool(tool_name)
            if not tool_function:
                observation = f"错误:未找到名为 '{tool_name}' 的工具。"
            else:
                observation = tool_function(tool_input) # 调用真实工具
            print(f"👀 观察: {observation}")
            
            # 将本轮的Action和Observation添加到历史记录中
            self.history.append(f"Action: {action}")
            self.history.append(f"Observation: {observation}")

        # 循环结束
        print("已达到最大步数，流程终止。")
        return None

    def _parse_output(self, text: str):
        """解析LLM的输出，提取Thought和Action。
        """
        # Thought: 匹配到 Action: 或文本末尾
        thought_match = re.search(r"Thought:\s*(.*?)(?=\nAction:|$)", text, re.DOTALL)
        # Action: 匹配到文本末尾
        action_match = re.search(r"Action:\s*(.*?)$", text, re.DOTALL)
        thought = thought_match.group(1).strip() if thought_match else None
        action = action_match.group(1).strip() if action_match else None
        return thought, action

    def _parse_action(self, action_text: str):
        """解析Action字符串，提取工具名称和输入。
        """
        match = re.match(r"(\w+)\[(.*)\]", action_text, re.DOTALL)
        if match:
            return match.group(1), match.group(2)
        return None, None

### 2.2.5 输出解析器
LLM 返回的是纯文本，我们需要从中精确地提取出`Thought`和`Action`。这是通过几个辅助解析函数完成的，它们通常使用正则表达式来实现。
- `_parse_output`： 负责从LLM的完整响应中分离出`Thought`和`Action`两个主要部分。
- `_parse_action`： 负责进一步解析`Action`字符串，例如从 `Search[华为最新手机]` 中提取出工具名 `Search` 和工具输入 `华为最新手机`。


### 2.2.6 ReAct 小结
#### 主要特点
| 特点 | 说明 |
| :--- | :--- |
| **高可解释性** | ReAct 最大的优点之一就是透明。通过 `Thought` 链，我们可以清晰地看到智能体每一步的“心路历程”——它为什么会选择这个工具，下一步又打算做什么。这对于理解、信任和调试智能体的行为至关重要。 |
| **动态规划与纠错能力** | 与一次性生成完整计划的范式不同，ReAct 是“走一步，看一步”。它根据每一步从外部世界获得的 `Observation` 来动态调整后续的 `Thought` 和 `Action`。如果上一步的搜索结果不理想，它可以在下一步中修正搜索词，重新尝试。 |
| **工具协同能力** | ReAct 范式天然地将大语言模型的推理能力与外部工具的执行能力结合起来。LLM 负责运筹帷幄（规划和推理），工具负责解决具体问题（搜索、计算），二者协同工作，突破了单一 LLM 在知识时效性、计算准确性等方面的固有局限。 |

#### 固有局限性
| 局限性 | 说明 |
| :--- | :--- |
| **对LLM自身能力的强依赖** | ReAct 流程的成功与否，高度依赖于底层 LLM 的综合能力。如果 LLM 的逻辑推理能力、指令遵循能力或格式化输出能力不足，就很容易在 `Thought` 环节产生错误的规划，或者在 `Action` 环节生成不符合格式的指令，导致整个流程中断。 |
| **执行效率问题** | 由于其循序渐进的特性，完成一个任务通常需要多次调用 LLM。每一次调用都伴随着网络延迟和计算成本。对于需要很多步骤的复杂任务，这种串行的“思考-行动”循环可能会导致较高的总耗时和费用。 |
| **提示词的脆弱性** | 整个机制的稳定运行建立在一个精心设计的提示词模板之上。模板中的任何微小变动，甚至是用词的差异，都可能影响 LLM 的行为。此外，并非所有模型都能持续稳定地遵循预设的格式，这增加了在实际应用中的不确定性。 |
| **可能陷入局部最优** | 步进式的决策模式意味着智能体缺乏一个全局的、长远的规划。它可能会因为眼前的 `Observation` 而选择一个看似正确但长远来看并非最优的路径，甚至在某些情况下陷入“原地打转”的循环中。 |

#### 调试技巧
| 技巧 | 说明 |
| :--- | :--- |
| **检查完整的提示词** | 在每次调用 LLM 之前，将最终格式化好的、包含所有历史记录的完整提示词打印出来。这是追溯 LLM 决策源头的最直接方式。 |
| **分析原始输出** | 当输出解析失败时（例如，正则表达式没有匹配到 `Action`），务必将 LLM 返回的原始、未经处理的文本打印出来。这能帮助你判断是 LLM 没有遵循格式，还是你的解析逻辑有误。 |
| **验证工具的输入与输出** | 检查智能体生成的 `tool_input` 是否是工具函数所期望的格式，同时也要确保工具返回的 `observation` 格式是智能体可以理解和处理的。 |
| **调整提示词中的示例 (Few-shot Prompting)** | 如果模型频繁出错，可以在提示词中加入一两个完整的“Thought-Action-Observation”成功案例，通过示例来引导模型更好地遵循你的指令。 |
| **尝试不同的模型或参数** | 更换一个能力更强的模型，或者调整 `temperature` 参数（通常设为0以保证输出的确定性），有时能直接解决问题。 |

## 2.3 Plan-and-Solve
思维链在处理多步骤、复杂问题时容易“偏离轨道”的问题，所以我们又回来了：
1. **规划阶段 (Planning Phase)**： 首先，智能体会接收用户的完整问题。它的第一个任务不是直接去解决问题或调用工具，而是 **将问题分解，并制定出一个清晰、分步骤的行动计划**。这个计划本身就是一次大语言模型的调用产物。
2. **执行阶段 (Solving Phase)**： 在获得完整的计划后，智能体进入执行阶段。它会 **严格按照计划中的步骤，逐一执行**。每一步的执行都可能是一次独立的 LLM 调用，或者是对上一步结果的加工处理，直到计划中的所有步骤都完成，最终得出答案。
说白了就是递归成子任务的一次规划执行。适用于那些结构性强、可以被清晰分解的复杂任务，例如：
- **多步数学应用题**：需要先列出计算步骤，再逐一求解。
- **需要整合多个信息源的报告撰写**：需要先规划好报告结构（引言、数据来源A、数据来源B、总结），再逐一填充内容。
- **代码生成任务**：需要先构思好函数、类和模块的结构，再逐一实现。
### 2.3.1 规划阶段
为了凸显 Plan-and-Solve 范式在结构化推理任务上的优势，他们将不使用工具的方式，而是通过提示词的设计，完成一个推理任务。

规划阶段的目标是让大语言模型接收原始问题，并输出一个清晰、分步骤的行动计划。这个计划必须是结构化的，以便我们的代码可以轻松解析并逐一执行。因此，我们设计的提示词需要明确地告诉模型它的角色和任务，并给出一个输出格式的范例。

In [ ]:
PLANNER_PROMPT_TEMPLATE = """
你是一个顶级的AI规划专家。你的任务是将用户提出的复杂问题分解成一个由多个简单步骤组成的行动计划。
请确保计划中的每个步骤都是一个独立的、可执行的子任务，并且严格按照逻辑顺序排列。
你的输出必须是一个Python列表，其中每个元素都是一个描述子任务的字符串。

问题: {question}

请严格按照以下格式输出你的计划,```python与```作为前后缀是必要的:
```python
["步骤1", "步骤2", "步骤3", ...]
```
"""

In [ ]:
class Planner:
    def __init__(self, llm_client):
        self.llm_client = llm_client

    def plan(self, question: str) -> list[str]:
        """
        根据用户问题生成一个行动计划。
        """
        prompt = PLANNER_PROMPT_TEMPLATE.format(question=question)
        
        # 为了生成计划，我们构建一个简单的消息列表
        messages = [{"role": "user", "content": prompt}]
        
        print("--- 正在生成计划 ---")
        # 使用流式输出来获取完整的计划
        response_text = self.llm_client.think(messages=messages) or ""
        
        print(f"✅ 计划已生成:\n{response_text}")
        
        # 解析LLM输出的列表字符串
        try:
            # 找到```python和```之间的内容
            plan_str = response_text.split("```python")[1].split("```")[0].strip()
            # 使用ast.literal_eval来安全地执行字符串，将其转换为Python列表
            plan = ast.literal_eval(plan_str)
            return plan if isinstance(plan, list) else []
        except (ValueError, SyntaxError, IndexError) as e:
            print(f"❌ 解析计划时出错: {e}")
            print(f"原始响应: {response_text}")
            return []
        except Exception as e:
            print(f"❌ 解析计划时发生未知错误: {e}")
            return []

### 2.3.2 执行器与状态管理
在规划器 (`Planner`) 生成了清晰的行动蓝图后，我们就需要一个执行器 (`Executor`) 来逐一完成计划中的任务。执行器不仅负责调用大语言模型来解决每个子问题，还承担着一个至关重要的角色：**状态管理**。它必须记录每一步的执行结果，并将其作为上下文提供给后续步骤，确保信息在整个任务链条中顺畅流动

执行器的提示词与规划器不同。它的目标不是分解问题，而是 **在已有上下文的基础上，专注解决当前这一个步骤**。因此，提示词需要包含以下关键信息：

- **原始问题**： 确保模型始终了解最终目标。
- **完整计划**： 让模型了解当前步骤在整个任务中的位置。
- **历史步骤与结果**： 提供至今为止已经完成的工作，作为当前步骤的直接输入。
- **当前步骤**： 明确指示模型现在需要解决哪一个具体任务。

In [ ]:
EXECUTOR_PROMPT_TEMPLATE = """
你是一位顶级的AI执行专家。你的任务是严格按照给定的计划，一步步地解决问题。
你将收到原始问题、完整的计划、以及到目前为止已经完成的步骤和结果。
请你专注于解决“当前步骤”，并仅输出该步骤的最终答案，不要输出任何额外的解释或对话。

# 原始问题:
{question}

# 完整计划:
{plan}

# 历史步骤与结果:
{history}

# 当前步骤:
{current_step}

请仅输出针对“当前步骤”的回答:
"""

In [ ]:
class Executor:
    def __init__(self, llm_client):
        self.llm_client = llm_client

    def execute(self, question: str, plan: list[str]) -> str:
        """
        根据计划，逐步执行并解决问题。
        """
        history = "" # 用于存储历史步骤和结果的字符串
        
        print("\n--- 正在执行计划 ---")
        
        for i, step in enumerate(plan):
            print(f"\n-> 正在执行步骤 {i+1}/{len(plan)}: {step}")
            
            prompt = EXECUTOR_PROMPT_TEMPLATE.format(
                question=question,
                plan=plan,
                history=history if history else "无", # 如果是第一步，则历史为空
                current_step=step
            )
            
            messages = [{"role": "user", "content": prompt}]
            
            response_text = self.llm_client.think(messages=messages) or ""
            
            # 更新历史记录，为下一步做准备
            history += f"步骤 {i+1}: {step}\n结果: {response_text}\n\n"
            
            print(f"✅ 步骤 {i+1} 已完成，结果: {response_text}")

        # 循环结束后，最后一步的响应就是最终答案
        final_answer = response_text
        return final_answer

In [ ]:
class PlanAndSolveAgent:
    def __init__(self, llm_client):
        """
        初始化智能体，同时创建规划器和执行器实例。
        """
        self.llm_client = llm_client
        self.planner = Planner(self.llm_client)
        self.executor = Executor(self.llm_client)

    def run(self, question: str):
        """
        运行智能体的完整流程:先规划，后执行。
        """
        print(f"\n--- 开始处理问题 ---\n问题: {question}")
        
        # 1. 调用规划器生成计划
        plan = self.planner.plan(question)
        
        # 检查计划是否成功生成
        if not plan:
            print("\n--- 任务终止 --- \n无法生成有效的行动计划。")
            return

        # 2. 调用执行器执行计划
        final_answer = self.executor.execute(question, plan)
        
        print(f"\n--- 任务完成 ---\n最终答案: {final_answer}")

## 2.4 Reflection
ReAct 和 Plan-and-Solve 范式中，智能体一旦完成了任务，其工作流程便告结束。然而，它们生成的初始答案，无论是行动轨迹还是最终结果，都可能存在谬误或有待改进之处。Reflection 机制的核心思想，正是为智能体引入一种 **事后（post-hoc）的自我校正循环**，使其能够像人类一样，审视自己的工作，发现不足，并进行迭代优化。

其核心工作流程可以概括为一个简洁的三步循环：**执行 -> 反思 -> 优化**。

1. **执行 (Execution)**：首先，智能体使用我们熟悉的方法（如 ReAct 或 Plan-and-Solve）尝试完成任务，生成一个初步的解决方案或行动轨迹。这可以看作是“初稿”。
2. **反思 (Reflection)**：接着，智能体进入反思阶段。它会调用一个独立的、或者带有特殊提示词的大语言模型实例，来扮演一个“评审员”的角色。这个“评审员”会审视第一步生成的“初稿”，并从多个维度进行评估，例如：
   - **事实性错误**：是否存在与常识或已知事实相悖的内容？
   - **逻辑漏洞**：推理过程是否存在不连贯或矛盾之处？
   - **效率问题**：是否有更直接、更简洁的路径来完成任务？
   - **遗漏信息**：是否忽略了问题的某些关键约束或方面？ 根据评估，它会生成一段结构化的 **反馈 (Feedback)**，指出具体的问题所在和改进建议。
3. **优化 (Refinement)**：最后，智能体将“初稿”和“反馈”作为新的上下文，再次调用大语言模型，要求它根据反馈内容对初稿进行修正，生成一个更完善的“修订稿”。

我们引入记忆管理机制，因为reflection通常对应着信息的存储和提取，如果上下文足够长的情况，想让“评审员”直接获取所有的信息然后进行反思往往会传入很多冗余信息。这一步实践我们主要完成 **代码生成与迭代优化**。
1. **存在明确的优化路径**：大语言模型初次生成的代码很可能是一个简单但效率低下的递归实现。
2. **反思点清晰**：可以通过反思发现其“时间复杂度过高”或“存在重复计算”的问题。
3. **优化方向明确**：可以根据反馈，将其优化为更高效的迭代版本或使用备忘录模式的版本。

### 2.4.1 Memory
- 使用一个列表 `records` 来按顺序存储每一次的行动和反思。
- `add_record` 方法负责向记忆中添加新的条目。
- `get_trajectory` 方法是核心，它将记忆轨迹“序列化”成一段文本，可以直接插入到后续的提示词中，为模型的反思和优化提供完整的上下文。
- `get_last_execution` 方便我们获取最新的“初稿”以供反思。


In [2]:
from typing import List, Dict, Any, Optional

class Memory:
    """
    一个简单的短期记忆模块，用于存储智能体的行动与反思轨迹。
    """

    def __init__(self):
        """
        初始化一个空列表来存储所有记录。
        """
        self.records: List[Dict[str, Any]] = []

    def add_record(self, record_type: str, content: str):
        """
        向记忆中添加一条新记录。

        参数:
        - record_type (str): 记录的类型 ('execution' 或 'reflection')。
        - content (str): 记录的具体内容 (例如，生成的代码或反思的反馈)。
        """
        record = {"type": record_type, "content": content}
        self.records.append(record)
        print(f"📝 记忆已更新，新增一条 '{record_type}' 记录。")

    def get_trajectory(self) -> str:
        """
        将所有记忆记录格式化为一个连贯的字符串文本，用于构建提示词。
        """
        trajectory_parts = []
        for record in self.records:
            if record['type'] == 'execution':
                trajectory_parts.append(f"--- 上一轮尝试 (代码) ---\n{record['content']}")
            elif record['type'] == 'reflection':
                trajectory_parts.append(f"--- 评审员反馈 ---\n{record['content']}")
        
        return "\n\n".join(trajectory_parts)

    def get_last_execution(self) -> Optional[str]:
        """
        获取最近一次的执行结果 (例如，最新生成的代码)。
        如果不存在，则返回 None。
        """
        for record in reversed(self.records):
            if record['type'] == 'execution':
                return record['content']
        return None

### 2.4.2 Reflection
Reflection 机制需要多个不同角色的提示词来协同工作。

In [3]:
INITIAL_PROMPT_TEMPLATE = """
你是一位资深的Python程序员。请根据以下要求，编写一个Python函数。
你的代码必须包含完整的函数签名、文档字符串，并遵循PEP 8编码规范。

要求: {task}

请直接输出代码，不要包含任何额外的解释。
"""

In [4]:
REFLECT_PROMPT_TEMPLATE = """
你是一位极其严格的代码评审专家和资深算法工程师，对代码的性能有极致的要求。
你的任务是审查以下Python代码，并专注于找出其在<strong>算法效率</strong>上的主要瓶颈。

# 原始任务:
{task}

# 待审查的代码:
```python
{code}
```

请分析该代码的时间复杂度，并思考是否存在一种<strong>算法上更优</strong>的解决方案来显著提升性能。
如果存在，请清晰地指出当前算法的不足，并提出具体的、可行的改进算法建议（例如，使用筛法替代试除法）。
如果代码在算法层面已经达到最优，才能回答“无需改进”。

请直接输出你的反馈，不要包含任何额外的解释。
"""

In [5]:
REFINE_PROMPT_TEMPLATE = """
你是一位资深的Python程序员。你正在根据一位代码评审专家的反馈来优化你的代码。

# 原始任务:
{task}

# 你上一轮尝试的代码:
{last_code_attempt}
评审员的反馈：
{feedback}

请根据评审员的反馈，生成一个优化后的新版本代码。
你的代码必须包含完整的函数签名、文档字符串，并遵循PEP 8编码规范。
请直接输出优化后的代码，不要包含任何额外的解释。
"""

In [6]:
class ReflectionAgent:
    def __init__(self, llm_client, max_iterations=3):
        self.llm_client = llm_client
        self.memory = Memory()
        self.max_iterations = max_iterations

    def run(self, task: str):
        print(f"\n--- 开始处理任务 ---\n任务: {task}")

        # --- 1. 初始执行 ---
        print("\n--- 正在进行初始尝试 ---")
        initial_prompt = INITIAL_PROMPT_TEMPLATE.format(task=task)
        initial_code = self._get_llm_response(initial_prompt)
        self.memory.add_record("execution", initial_code)

        # --- 2. 迭代循环:反思与优化 ---
        for i in range(self.max_iterations):
            print(f"\n--- 第 {i+1}/{self.max_iterations} 轮迭代 ---")

            # a. 反思
            print("\n-> 正在进行反思...")
            last_code = self.memory.get_last_execution()
            reflect_prompt = REFLECT_PROMPT_TEMPLATE.format(task=task, code=last_code)
            feedback = self._get_llm_response(reflect_prompt)
            self.memory.add_record("reflection", feedback)

            # b. 检查是否需要停止
            if "无需改进" in feedback:
                print("\n✅ 反思认为代码已无需改进，任务完成。")
                break

            # c. 优化
            print("\n-> 正在进行优化...")
            refine_prompt = REFINE_PROMPT_TEMPLATE.format(
                task=task,
                last_code_attempt=last_code,
                feedback=feedback
            )
            refined_code = self._get_llm_response(refine_prompt)
            self.memory.add_record("execution", refined_code)
        
        final_code = self.memory.get_last_execution()
        print(f"\n--- 任务完成 ---\n最终生成的代码:\n```python\n{final_code}\n```")
        return final_code

    def _get_llm_response(self, prompt: str) -> str:
        """一个辅助方法，用于调用LLM并获取完整的流式响应。"""
        messages = [{"role": "user", "content": prompt}]
        response_text = self.llm_client.think(messages=messages) or ""
        return response_text


### 2.4.3 Reflection 小结
尽管 Reflection 机制在提升任务解决质量上表现出色，但这种能力的获得并非没有代价。在实际应用中，我们需要权衡其带来的收益与相应的成本。

#### 主要成本
| 成本项 | 说明 |
| :--- | :--- |
| **模型调用开销增加** | 这是最直接的成本。每进行一轮迭代，至少需要额外调用两次大语言模型（一次用于反思，一次用于优化）。如果迭代多轮，API 调用成本和计算资源消耗将成倍增加。 |
| **任务延迟显著提高** | Reflection 是一个串行过程，每一轮的优化都必须等待上一轮的反思完成。这使得任务的总耗时显著延长，不适合对实时性要求高的场景。 |
| **提示工程复杂度上升** | 如我们的案例所示，Reflection 的成功在很大程度上依赖于高质量、有针对性的提示词。为“执行”、“反思”、“优化”等不同阶段设计和调试有效的提示词，需要投入更多的开发精力。 |

#### 核心收益
| 收益项 | 说明 |
| :--- | :--- |
| **解决方案质量的跃迁** | 最大的收益在于，它能将一个“合格”的初始方案，迭代优化成一个“优秀”的最终方案。这种从功能正确到性能高效、从逻辑粗糙到逻辑严谨的提升，在很多关键任务中是至关重要的。 |
| **鲁棒性与可靠性增强** | 通过内部的自我纠错循环，智能体能够发现并修复初始方案中可能存在的逻辑漏洞、事实性错误或边界情况处理不当等问题，从而大大提高了最终结果的可靠性。 |

#### 适用场景与策略建议
| 类别 | 内容 |
| :--- | :--- |
| **适用场景** | Reflection 机制是一种典型的“以成本换质量”的策略。它非常适合那些**对最终结果的质量、准确性和可靠性有极高要求，且对任务完成的实时性要求相对宽松**的场景。例如：<br>- 生成关键的业务代码或技术报告。<br>- 在科学研究中进行复杂的逻辑推演。<br>- 需要深度分析和规划的决策支持系统。 |
| **替代选择** | 反之，如果应用场景需要快速响应，或者一个“大致正确”的答案就已经足够，那么使用更轻量的 ReAct 或 Plan-and-Solve 范式可能会是更具性价比的选择。 |

## 2.5 课后习题
1. 本章介绍了三种经典的智能体范式:`ReAct`、`Plan-and-Solve` 和 `Reflection`。请分析:

   - 这三种范式在"思考"与"行动"的组织方式上有什么本质区别？
   - 如果要设计一个"智能家居控制助手"（需要控制灯光、空调、窗帘等多个设备，并根据用户习惯自动调节），你会选择哪种范式作为基础架构？为什么？
   - 是否可以将这三种范式进行组合使用？若可以，请尝试设计一个混合范式的智能体架构，并说明其适用场景。

---

- **ReAct（推理+行动交替）**：思考（Thought）与行动（Action）是交替进行的，每一步的思考都基于前一步的观察结果，然后立即执行一个行动，形成一个“思考→行动→观察→思考→行动→观察”的循环。推理指导行动，行动结果反馈给下一轮推理。
- **Plan-and-Solve（先规划后执行）**：将思考与行动分为两个阶段。首先进行一次性的全局“思考”（规划），生成完整的步骤序列；然后“行动”阶段按顺序执行这些步骤，执行过程中一般不再进行高层推理，除非引入重规划。
- **Reflection（执行后反思）**：先由执行者独立生成一个完整的输出（行动结果），然后引入一个反思者对该输出进行“思考”（评价和批评），再将反思结果反馈给执行者进行优化。思考和行动是串行的两阶段循环，但反思本身不直接与环境交互，而是基于已有输出进行元认知。

智能家居控制助手选择 **ReAct** 作为基础架构更合适。理由：
- 家庭环境状态动态变化（如有人进入房间、温度变化），需要**实时感知和即时决策**，ReAct 的交替结构允许模型根据当前观察（传感器数据、设备状态）逐步推理并采取行动。
- Plan-and-Solve 的静态计划难以应对环境动态性；Reflection 更适合离线优化（如生成固定流程），而非实时控制。
- ReAct 可以自然地将每一步推理与工具调用（设备控制 API）结合，例如：“观察到客厅光线不足 → 思考是否需要开灯 → 行动：打开客厅灯”。

一种混合范式的设计：**ReAct + Reflection + 计划引导**。
架构：
1. 用户指令进入后，首先由一个轻量级 Planner 生成一个高层计划（Plan-and-Solve 的思想），作为后续行为的指导蓝图。
2. 执行层采用 ReAct 循环，每一步推理和行动时都参考当前计划步骤，但允许根据实时观察动态调整。
3. 当一个子任务完成或遇到困难时，引入一个 Reflector 对已完成的行为序列和结果进行反思，评估是否偏离目标、效率如何，并生成改进建议。
4. 改进建议反馈给 Planner 或直接调整 ReAct 的后续行为。
适用场景：复杂的长期自治任务，如“智能家居日常节能管理”：需长期运行、动态变化、多个目标权衡（舒适 vs 节能），既有宏观计划（白天利用自然光，夜间低功耗模式），又需实时响应（有人回家提前开空调），还可周期性反思能耗数据优化策略。


2. 在4.2节的 `ReAct` 实现中，我们使用了正则表达式来解析大语言模型的输出（如 `Thought` 和 `Action`）。请思考:

   - 当前的解析方法存在哪些潜在的脆弱性？在什么情况下可能会失败？
   - 除了正则表达式，还有哪些更鲁棒的输出解析方案？
   - 尝试修改本章的代码，使用一种更可靠的输出格式，并对比两种方案的优缺点

---

当前正则方案（如 `Thought: (.*?)\nAction: (.*?)\n`）脆弱性体现在：
- **格式敏感**：模型输出必须严格匹配期望模式，多一个空格、换行符变化、大小写偏差、中英文标点混淆都可能导致解析失败。
- **内容干扰**：思考内容中可能包含类似“Action:”的字样，导致错误截断。
- **多行动/多步骤**：当模型在一次回复中生成多个 Thought-Action 对时，简单正则只能捕获第一个或最后一个，难以处理流式多轮。
- **模型幻觉**：模型可能忘记添加 `Action:` 前缀，或使用了 `Action Input:` 等其他格式，正则完全失效。
- **代码块包裹**：模型有时会将输出放在 markdown 代码块中，正则无法匹配。
更鲁棒的输出解析方案则为：
- **JSON 模式 / 结构化输出**：要求模型以 JSON 格式输出，如 `{"thought": "...", "action": "...", "action_input": "..."}`，然后使用 `json.loads()` 解析。可结合 OpenAI 的 function calling 或 JSON mode。鲁棒性远高于正则。
- **函数调用（Function Calling）**：直接利用大模型的工具调用 API，解析为结构化的函数调用对象，无需文本解析。
- **约束解码（Guided Decoding）**：使用类似 `guidance`、`lm-format-enforcer` 等库强制模型按照指定语法生成，从根本上保证格式正确。
- **后备修复**：正则解析失败时，尝试使用更灵活的匹配（如模糊搜索“Action”关键字），或用简单的 LLM 二次修正：“请将以下文本转换为 JSON 格式：...”。

| 方案 | 优点 | 缺点 |
|------|------|------|
| 正则表达式 | 简单轻量，无需特殊模型支持 | 极为脆弱，格式稍有偏差即失败；维护成本高 |
| JSON 结构输出 | 标准格式，易于解析和调试；支持复杂嵌套；大部分现代模型支持良好 | 模型仍可能输出非法 JSON；需增加容错处理；提示词更长 |
| Function Calling | 平台原生支持，鲁棒性最高；直接返回结构化对象 | 依赖特定 API；灵活性受限；非开源模型不一定具备 |


3. 工具调用是现代智能体的核心能力之一。基于4.2.2节的 `ToolExecutor` 设计，请完成以下扩展实践:

   - 为 `ReAct` 智能体添加一个"计算器"工具，使其能够处理复杂的数学计算问题（如"计算 `(123 + 456) × 789/ 12 = ?` 的结果"）
   - 设计并实现一个"工具选择失败"的处理机制:当智能体多次调用错误的工具或提供错误的参数时，系统应该如何引导它纠正？
   - 思考:如果可调用工具的数量增加到$50$个甚至$100$个，当前的工具描述方式是否还能有效工作？在可调用工具数量随业务需求显著增加时，从工程角度如何优化工具的组织和检索机制？

In [ ]:
import math

def calculator(expression: str) -> str:
    """
    安全的计算器工具，仅支持基本数学运算和数学函数。
    """
    # 安全限制：只允许数字、运算符、括号、空格和部分数学函数
    allowed_chars = set("0123456789+-*/().^% eEpPiIsqrtlnlog2sincostanabs")
    # 简单检查表达式安全性
    if not all(c in allowed_chars for c in expression):
        return "错误：表达式包含不允许的字符"
    try:
        # 使用 eval 存在风险，这里限定在安全环境，生产环境建议使用 numexpr 或 ast 解析
        result = eval(expression, {"__builtins__": None}, {
            "sqrt": math.sqrt,
            "sin": math.sin,
            "cos": math.cos,
            "tan": math.tan,
            "log": math.log,
            "log2": math.log2,
            "abs": abs,
            "pi": math.pi,
            "e": math.e
        })
        return str(result)
    except Exception as e:
        return f"计算错误：{str(e)}"

# 工具描述
calculator_tool = {
    "name": "Calculator",
    "description": "执行数学计算。输入为一个数学表达式，例如 '(123 + 456) * 789 / 12'。支持加减乘除、括号、幂运算及sqrt,sin,cos,tan,log,abs等函数。",
    "func": calculator
}

In [ ]:
class RobustToolExecutor:
    # “错误恢复引导”机制，当连续多次出现工具调用错误时触发：
    def __init__(self, tools, max_retries=3):
        self.tools = {t["name"]: t for t in tools}
        self.error_history = []  # 记录错误历史
        self.max_retries = max_retries

    def execute(self, action, action_input):
        if action not in self.tools:
            self.error_history.append(f"未找到工具 '{action}'")
            return f"错误：没有名为 '{action}' 的工具。可用工具：{list(self.tools.keys())}"
        try:
            result = self.tools[action]["func"](action_input)
            self.error_history.clear()  # 成功后清空错误历史
            return result
        except Exception as e:
            self.error_history.append(f"工具 '{action}' 调用失败: {str(e)}")
            return f"工具调用失败：{str(e)}。请检查参数格式，参考工具描述。"
    
    def get_guidance(self):
        """
        当连续失败达到阈值时，生成引导提示，帮助模型纠正。
        """
        if len(self.error_history) >= self.max_retries:
            guidance = "你已连续多次调用失败，错误记录如下：\n"
            guidance += "\n".join(f"- {e}" for e in self.error_history)
            guidance += "\n请仔细阅读工具描述，检查工具名称和参数格式。可尝试使用相近的工具或分解操作。"
            self.error_history.clear()
            return guidance
        return None

当前工具描述方式将所有工具描述放入提示词，可能导致：
- 上下文窗口超限（尤其是工具描述很长时）
- 模型注意力分散，选择准确率下降
- 推理成本升高（输入 tokens 增加）

**工程优化方案**：
- **工具检索式分派**：采用两阶段方法。第一阶段，使用轻量级检索模型（如基于嵌入相似度）根据用户 query 从工具库中召回 top-k（如5~10）个相关工具；第二阶段，仅将这些工具的描述注入提示词。这类似于“工具路由器”模式。
- **工具分组与层级命名**：将工具按领域分组（如 `math.*`，`database.*`，`device.*`），模型首先选择领域，再在该领域内选择具体工具，可分层提示。
- **工具描述摘要与动态示例**：为每个工具提供简短单行描述用于第一阶段筛选，完整描述仅在工具被选中后动态加载。
- **离线索引与缓存**：对工具描述建立向量索引（如用 sentence-transformers），每次请求实时检索相关工具。
- **工具使用历史的上下文学习**：缓存类似任务的工具选择序列，作为 few-shot 示例嵌入，提高选择效率。

4. `Plan-and-Solve` 范式将任务分解为"规划"和"执行"两个阶段。请深入分析:

   - 在4.3节的实现中，规划阶段生成的计划是"静态"的（一次性生成，不可修改）。如果在执行过程中发现某个步骤无法完成或结果不符合预期，应该如何设计一个"动态重规划"机制？
   - 对比 `Plan-and-Solve` 与 `ReAct`:在处理"预订一次从北京到上海的商务旅行（包括机票、酒店、租车）"这样的任务时，哪种范式更合适？为什么？
   - 尝试设计一个"分层规划"系统:先生成高层次的抽象计划，然后针对每个高层步骤再生成详细的子计划。这种设计有什么优势？

---

当执行过程中某步骤失败或结果不符合预期时，引入动态重规划机制：
- **执行监控**：每个步骤执行后，使用一个评估器（可以是基于规则的检查或轻量 LLM）判断该步骤是否成功，输出置信度或状态标记（成功/失败/部分成功）。
- **触发重规划**：若步骤失败，系统暂停当前计划执行，收集已完成步骤的状态和失败原因。
- **增量重规划**：调用 Planner 模块，输入原始任务、已成功完成的步骤、失败的步骤及原因、当前环境状态，生成一个新的“尾部计划”替代原计划中失败及后续步骤，保持已完成部分不变。这比重新生成整个计划效率更高。
- **全局重规划**：如果增量重规划也无法解决（如多次失败），则退回起点进行全局重规划。
- **输出拼接**：将原成功步骤的输出与新计划的输出无缝衔接。

对于“预订一次从北京到上海的商务旅行（包括机票、酒店、租车）”，**ReAct 更为合适**，理由：
- **依赖关系复杂**：酒店入住时间依赖航班到达，租车取车地点可能依赖酒店位置。这些依赖无法预先完全确定（如航班可选班次多，价格实时变动），需要边查边调整。
- **环境交互频繁**：每次查询（查航班、查酒店、查租车）返回的信息会影响后续选择，ReAct 的“观察→思考→行动”模式更适合这种逐步决策。
- **错误恢复**：如果某航司机票售罄，ReAct 可以立即根据观察调整思路，尝试其他航班，而静态计划需要显式重规划。
Plan-and-Solve 在子任务相对独立、可提前完全规划的场景（如表格数据批量处理）更优，但旅行预订中不确定性大，因此 ReAct 更具优势。

系统分为两层：
- **高层规划器（宏计划）**：生成抽象步骤，如“1. 预订往返机票；2. 预订两晚酒店；3. 预订租车服务”。
- **低层规划器（微计划）**：为每个高层步骤动态生成详细的子计划。例如针对“预订往返机票”生成子步骤：“①查询6月20日北京→上海航班；②选择符合预算和时间窗的航班；③填写乘客信息并预订；④查询6月22日上海→北京返程航班；⑤预订返程”。
**优势**：
- **降低单次规划复杂度**：每个规划只处理有限上下文，降低模型认知负荷，提高计划质量。
- **灵活性和可复用性**：高层计划不涉及具体操作，可复用；低层计划可根据实时信息动态生成，适应环境变化。
- **模块化与可扩展**：复杂任务可插入新的子任务，重规划只需在受影响的局部层进行，不影响整体结构。
- **便于人机协作**：用户可在高层计划阶段审核调整宏观步骤，精细执行交由系统自动完成。


5. `Reflection` 机制通过"执行-反思-优化"循环来提升输出质量。请思考:

   - 在4.4节的代码生成案例中，不同阶段使用的是同一个模型。如果使用两个不同的模型（例如，用一个更强大的模型来做反思，用一个更快的模型来做执行），会带来什么影响？
   - `Reflection` 机制的终止条件是"反馈中包含 **无需改进**"或"达到最大迭代次数"。这种设计是否合理？能否设计一个更智能的终止条件？
   - 假设你要搭建一个"学术论文写作助手"，它能够生成初稿并不断优化论文内容。请设计一个多维度的Reflection机制，从段落逻辑性、方法创新性、语言表达、引用规范等多个角度进行反思和改进。

---

对于强模型反思，弱模型执行：
这背后是经典的“**弱模型生成，强模型验证**”思想，在许多研究和实践中都得到了验证。
*   **核心优势**：**质量与成本的平衡**。强模型（如GPT Pro、Claude Opus）的批判性思维更强，能指出逻辑漏洞和优化空间，提升反思深度。而弱模型（如Kimi、Qwen-8B）成本低、速度快，适合快速执行，整体上实现了效果与效率的平衡。
*   **潜在风险**：可能出现“**建议过载**”或“**能力错配**”。强模型提出的建议可能超出弱模型的能力范围，导致执行模型无法有效跟随，或者陷入不断尝试但无法达到要求的循环。
*   **适用场景**：适合对**成本敏感**且**任务有明确标准**的场景，如大批量代码审查、内容审核等。

而对于弱模型反思，强模型执行：
这个思路其实更接近“**专业分工，各司其职**”。
*   **核心优势**：**聚焦与稳定**。用专门的“评审”（弱模型）做反思，可以避免“**自我偏见**”——同一个模型既当运动员又当裁判，往往会认同自己的错误。不同模型能提供**多样化的视角**。让最强的模型专注于执行，能最大化其核心能力，避免因反思带来的额外延迟和成本。
*   **潜在风险**：**“评审”水平可能不够**。如果反思模型太弱，可能无法发现强模型执行中的深层问题，导致反思环节流于形式。
*   **实践案例**：这种模式已有不少实践，例如用**Kimi K2.5**作为“指挥官”去调度**Claude Opus**。在多模型协作中，也让不同的模型各展所长。
*   **适用场景**：适合**任务极其复杂**、对执行质量要求极高且预算充足的场景。比如顶尖的科研分析、复杂的战略规划等。

终止条件（“无需改进”或最大迭代次数）有其合理性，但存在局限：
- “无需改进”依赖模型自我评价，可能过早自信（模型自满）或过于苛刻（永无止境）。
- 最大迭代次数是粗暴兜底，可能浪费算力或未达到最优即终止。
**更智能的终止条件**：
- **多指标收敛判断**：除自我评价外，引入外部客观指标（如代码执行通过率、BLEU、ROUGE等），当指标提升幅度低于阈值时停止。
- **滑动窗口停滞检测**：跟踪最近 k 次反思的改进量，若连续无显著改进，终止。
- **反思-生成一致性评分**：由反思模型对改进前后输出进行成对比较打分，若得分差低于阈值，终止。
- **成本-收益权衡**：设定预算（时间或金钱），在接近预算上限时采用宽松终止策略。

**设计思路**：将反思分解为多个独立维度，每个维度由一个专门的反思提示词负责，最终汇总为一个结构化改进计划。可并行执行各维度的反思，以提高效率。
**维度设计**：
- **段落逻辑性**：检查论点之间衔接、论证链完整性、主题句和支撑句对应关系。反思输出：逻辑断裂点、建议的重排或补充。
- **方法创新性**：对比常见基线方法，评述本文方法的独创性和贡献清晰度。反思输出：创新点突出建议、与相关工作的区分说明。
- **语言表达**：关注语法、拼写、冗余、术语一致性、可读性。反思输出：逐段语言润色建议。
- **引用规范**：检查引用格式、遗漏关键文献、引用准确性。反思输出：缺失引用列表、格式修正点。
- **实验严谨性**（若含实验）：检查实验设置、指标选择、显著性检验。反思输出：实验改进建议。
- **整体结构**：标题、摘要、章节逻辑流。反思输出：结构调整建议。
**流程**：
1. 执行者生成初稿。
2. 多维度反思器并行评审，分别生成评审意见。
3. 汇总器将多维度意见合并，去重并排序优先级，形成统一的修改指导。
4. 执行者基于指导进行修改，进入下一轮循环。


6. 提示词工程是影响智能体最终效果的关键技术。本章展示了多个精心设计的提示词模板。请分析:

   - 对比4.2.3节的 `ReAct` 提示词和4.3.2节的 `Plan-and-Solve` 提示词，它们显然存在结构设计上的明显不同，这些差异是如何服务于各自范式的核心逻辑的？
   - 在4.4.3节的 `Reflection` 提示词中，我们使用了"你是一位极其严格的代码评审专家"这样的角色设定。尝试修改这个角色设定（如改为"你是一位注重代码可读性的开源项目维护者"），观察输出结果的变化，并总结角色设定对智能体行为的影响。
   - 在提示词中加入 `few-shot` 示例往往能显著提升模型对特定格式的遵循能力。请为本章的某个智能体尝试添加 `few-shot` 示例，并对比其效果。

---

- **ReAct 提示词**：需要详细定义“可用工具”及其描述，并给出“你必须遵循以下格式：Thought: ... Action: ... Observation: ...”的严格交替格式要求。重点在于 **实时交互** 和 **工具使用规范**，引导模型在每一步都显式推理当前状态并选择行动，强调 Observation 是外部反馈。这种结构为“推理-行动”循环提供每一步的模板。
- **Plan-and-Solve 提示词**：不需要工具调用格式（或很少），而是要求“首先分析问题并制定完整计划，然后逐步执行”。重点在于 **规划能力** 和 **分步执行**。它强调“计划”作为一个独立的、前置的输出块，格式可能为“Plan: 1. ... 2. ...”，然后“Step 1: ...”。这种结构促使模型进行全局前置思考，而不是边做边想。
- **核心差异**：ReAct 提示词通过交替格式强制进行 **交错推理**；Plan-and-Solve 提示词通过分离“计划”和“执行”部分强制进行 **先行规划**。两种结构都通过输出格式约束塑造模型的思考模式。

In [ ]:
FEWSHOT_EXAMPLES = """
示例1:
问题: 查一下今天杭州的天气，并告诉我是否适合户外跑步。
思考: 我需要先获取杭州今天的天气信息。
行动: GetWeather
行动输入: 杭州
观察: 杭州今天中雨，温度18-22℃，空气质量优。
思考: 天气为中雨，跑步会淋湿，不适合户外跑步。
行动: Finished
行动输入: 今天杭州中雨，不适合户外跑步。

示例2:
问题: 计算 (123+456)*789/12 的结果
思考: 我需要使用计算器工具来计算这个数学表达式。
行动: Calculator
行动输入: (123+456)*789/12
观察: 38092.25
思考: 计算完成，结果是38092.25。
行动: Finished
行动输入: 38092.25
"""

- **无 few-shot**：模型可能偶尔输出格式错误，或思考与行动不匹配，或提前终止。
- **有 few-shot**：输出格式遵循度显著提高，尤其对工具调用格式和 Finished 时机有明确参照，减少了解析失败次数。但会增加 prompt 长度，对于复杂场景需精心挑选覆盖边界情况的示例。



7. 某电商初创公司现在希望使用"客服智能体"来代替真人客服实现降本增效，它需要具备以下功能:

   a. 理解用户的退款申请理由

   b. 查询用户的订单信息和物流状态

   c. 根据公司政策智能地判断是否应该批准退款

   d. 生成一封得体的回复邮件并发送至用户邮箱

   e. 如果判断决策存在一定争议（自我置信度低于阈值），能够进行自我反思并给出更审慎的建议

   此时作为该产品的负责人:
   - 你会选择本章的哪种范式（或哪些范式的组合）作为系统的核心架构？
   - 这个系统需要哪些工具？请列出至少3个工具及其功能描述。
   - 如何设计提示词来确保智能体的决策既符合公司利益，又能保持对用户的友好态度？
   - 这个产品上线后可能面临哪些风险和挑战？如何通过技术手段来降低这些风险？

---

选择 **ReAct + Reflection 组合**作为核心架构。
- 主循环采用 **ReAct**：处理用户退款申请时，智能体需要多步查询和推理（查订单、查物流、查政策），并根据查询结果做出是否批准的决策，其间必须实时根据工具返回的信息调整思考，因此 ReAct 的交替推理-行动模式最合适。
- 当决策不确定性高（自我置信度低于阈值）时，触发 **Reflection** 分支：将当前决策与理由输入反思模块，让其从多角度（公司政策、用户体验、法律风险）审查，并给出更审慎的建议，最终输出优化后的决策。

| 工具名称 | 输入 | 返回 |
| :--- | :--- | :--- |
| **订单信息查询工具 (QueryOrder)** | 订单号 | 订单详情（商品、金额、下单时间、支付方式、当前状态、用户ID等） |
| **物流状态查询工具 (QueryLogistics)** | 订单号 | 物流状态（已发货/运输中/已签收/退回中），详细的物流轨迹 |
| **退款政策匹配工具 (CheckRefundPolicy)** | 订单状态、退款理由关键词、商品类别等 | 适用退款政策条款、条件限制（如是否在7天无理由范围内、是否特殊商品）、建议处理方式 |
| **邮件发送工具 (SendEmail)** | 收件人邮箱、邮件主题、邮件正文 | 发送成功或失败的确认 |

- **角色设定**：“你是一位专业的电商客服专家，既需遵循公司退款政策维护公司合法权益，又要以友好、共情的方式为用户解决问题。”
- **决策原则嵌入**：明确列出决策需遵循的优先级——①遵守法律和平台规定；②公司退款政策；③用户在合理范围内的满意度；④避免恶意退款损失。
- **语气风格指引**：要求输出对用户的邮件时，语气需“温暖、尊重、体现同理心，即使拒绝退款也需清晰说明理由并给予替代方案（如换货、优惠券）。”
- **置信度要求**：要求模型对每次决策输出一个0-100的置信度评分，并指明如果低于阈值（如80）应主动触发反思。


| 类别 | 具体内容 |
| :--- | :--- |
| **风险与挑战** | 误判风险：AI 错误批准恶意退款，或错误拒绝合理退款，导致公司损失或用户投诉、公关危机。 |
|  | 政策理解偏差：LLM 对公司复杂多变的退款政策理解不足，做出不一致决策。 |
|  | 被攻击风险：用户通过精心设计的 prompt 注入或 social engineering 话术诱导智能体违规退款。 |
|  | 责任归属：AI 自主决策造成纠纷，责任界定困难。 |
|  | 用户体验风险：机器回复冷漠、模板化，导致用户不满升级为舆情事件。 |
| **技术降险措施** | 人工审核兜底：对高金额、高争议度、低置信度的决策自动转入人工队列。 |
|  | 政策向量知识库：将公司政策、法规条款存入向量数据库，Retrieval-Augmented Generation（RAG）增强决策依据的准确性和时效性。 |
|  | 输入/输出安全防护：部署内容安全审查模块，检测 prompt 注入和对抗性输入；对输出邮件进行合规性和语气审核。 |
|  | A/B 灰度发布与监控：先在低风险场景小流量上线，实时监控退款纠纷率、用户满意度、决策置信度分布等指标，逐步放量。 |
|  | 可解释性与日志审计：记录每一步推理和工具调用，所有决策均可追溯，便于审计和责任分析。 |